# Wire Response Visualization

This notebook demonstrates visualization of the extracted wire response kernels and their diffusion properties.

In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
import sys

# Add parent directory to path for imports
sys.path.append('/home/oalterka/desktop_linux/JAXTPC')

# Import our response processing modules
from tools.responses.diffusion_kernels import (
    create_diffusion_kernel_array,
    calculate_wire_count
)
from tools.responses.visualization import (
    visualize_kernel,
    visualize_diffusion_progression,
    create_parameter_sweep_gif
)

# Set matplotlib parameters for better display
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (10, 8)

print("Libraries imported successfully!")

## Load Diffusion Kernels

Load the pre-extracted kernel files and create diffusion kernel arrays.

In [ ]:
# Configuration
planes = ['U', 'V', 'Y']
num_s = 16  # Number of diffusion levels
wire_spacing = 0.1
time_spacing = 0.5

# Create diffusion kernel arrays
print("Loading kernels and creating diffusion arrays...\n")

DKernels = create_diffusion_kernel_array(
    planes=planes,
    num_s=num_s,
    kernel_dir='.',  # Current directory
    wire_spacing=wire_spacing,
    time_spacing=time_spacing
)

print("\n✓ Kernels loaded successfully!")
print(f"\nAvailable planes: {list(DKernels.keys())}")

## Visualize Original Kernels

Display the original (non-diffused) kernels for each wire plane.

In [ ]:
# Visualize original kernels (s=0) for each plane
for plane in DKernels.keys():
    DKernel, linear_s, kernel_shape, x_coords, y_coords = DKernels[plane]
    
    # Extract original kernel (s=0)
    original_kernel = DKernel[0]
    
    # Create visualization
    fig, ax = visualize_kernel(
        original_kernel, 
        x_coords, 
        y_coords, 
        plane=plane,
        figsize=(8, 8)
    )
    
    # Add some statistics
    center_idx = (kernel_shape[0]//2, kernel_shape[1]//2)
    center_value = original_kernel[center_idx]
    max_value = np.max(original_kernel)
    min_value = np.min(original_kernel)
    
    ax.text(0.02, 0.98, f'Center value: {center_value:.3f}\nMax: {max_value:.3f}\nMin: {min_value:.3f}',
            transform=ax.transAxes, fontsize=10, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    plt.tight_layout()
    plt.show()

## Visualize Diffusion Progression

Show how the kernels change with increasing diffusion parameter s.

In [ ]:
# Show diffusion progression for each plane
for plane in DKernels.keys():
    DKernel, linear_s, kernel_shape, x_coords, y_coords = DKernels[plane]
    
    fig = visualize_diffusion_progression(
        DKernel, 
        linear_s, 
        x_coords, 
        y_coords, 
        plane=plane,
        figsize=(15, 3)
    )
    
    plt.tight_layout()
    plt.show()

## Compare Planes

Compare the three wire planes side by side.

In [ ]:
# Compare all three planes at s=0
from tools.responses.visualization import actual_to_paper_log10

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Convert all kernels first to find global limits
all_kernels_log10 = []
for plane in ['U', 'V', 'Y']:
    if plane in DKernels:
        DKernel, _, _, _, _ = DKernels[plane]
        kernel = DKernel[0]
        kernel_log10 = actual_to_paper_log10(np.array(kernel))
        all_kernels_log10.append(kernel_log10)

# Find maximum absolute value for symmetric colorbar
vmax = max(np.max(np.abs(k)) for k in all_kernels_log10)
vmin = -vmax

for i, (plane, kernel_log10) in enumerate(zip(['U', 'V', 'Y'], all_kernels_log10)):
    if plane in DKernels:
        DKernel, linear_s, kernel_shape, x_coords, y_coords = DKernels[plane]
        
        # Create image with symmetric colorbar
        im = axes[i].imshow(kernel_log10, aspect='auto',
                           extent=[x_coords[0], x_coords[-1], y_coords[0], y_coords[-1]],
                           cmap='RdBu_r', origin='lower', vmin=vmin, vmax=vmax)
        
        # Formatting
        axes[i].set_xlabel('Wire Number')
        axes[i].set_ylabel('Time [μs]' if i == 0 else '')
        axes[i].set_title(f'{plane} Plane')
        axes[i].grid(True, alpha=0.3)
        axes[i].axvline(x=0, color='black', linestyle='--', alpha=0.5)
        axes[i].axhline(y=0, color='black', linestyle='--', alpha=0.5)
        
        # Individual colorbar (all will have same scale)
        cbar = plt.colorbar(im, ax=axes[i], pad=0.02)
        cbar.set_label('Log 10″', rotation=270, labelpad=15)

plt.suptitle('Wire Response Kernels - All Planes (s=0)', fontsize=14)
plt.tight_layout()
plt.show()

print(f"Colorbar range: [{vmin:.1f}, {vmax:.1f}] (symmetric around 0)")

## Visualize Interpolation Steps

Show the detailed interpolation process with the 2x2 layout visualization.

In [ ]:
# Import the visualization function
from tools.responses.visualization import visualize_interpolation_steps

# Show interpolation steps
fig = visualize_interpolation_steps(DKernels, plane='U',
                                  s_observed=0.3, w_offset=0.25, t_offset=0.15,
                                  verbose=True)
plt.show()

# Create GIF animations for different parameter sweeps using the detailed visualization
# Note: This will create GIF files in the current directory

In [ ]:
# 1. Diffusion parameter (s) sweep
print("Creating diffusion parameter sweep animation...")
create_parameter_sweep_gif(
    visualize_interpolation_steps,
    DKernels,
    plane='Y',
    parameter='s_observed',
    param_range=(0, 1),
    fixed_params={'w_offset': 0.25, 't_offset': 0.15},
    output_filename='gifs/diffusion_sweep_detailed_Y.gif',
    n_frames=30,
    fps=10
)
print("✓ Created diffusion_sweep_detailed_Y.gif")

In [ ]:
# 2. Wire offset sweep
print("Creating wire offset sweep animation...")
create_parameter_sweep_gif(
    visualize_interpolation_steps,
    DKernels,
    plane='Y',
    parameter='w_offset',
    param_range=(0, 0.99),
    fixed_params={'s_observed': 0.3, 't_offset': 0.15},
    output_filename='gifs/wire_offset_sweep_detailed_Y.gif',
    n_frames=60,
    fps=20
)
print("✓ Created wire_offset_sweep_detailed_Y.gif")

In [ ]:
# 3. Time offset sweep
print("Creating time offset sweep animation...")
create_parameter_sweep_gif(
    visualize_interpolation_steps,
    DKernels,
    plane='Y',
    parameter='t_offset',
    param_range=(0, 0.49),
    fixed_params={'s_observed': 0.3, 'w_offset': 0.25},
    output_filename='gifs/time_offset_sweep_detailed_Y.gif',
    n_frames=20,
    fps=10
)
print("✓ Created time_offset_sweep_detailed_Y.gif")